In [1]:
import qiskit
import qiskit_aer

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

print("Qiskit version:", qiskit.__version__)

simulator = AerSimulator()

print("Quantum simulator initialized!")

Qiskit version: 2.6.2
Quantum simulator initialized!


In [24]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt

class BB84Protocol:
    """
    BB84 Quantum Key Distribution Protocol Implementation
    
    Steps:
    1. Alice generates random bits and bases
    2. Alice encodes qubits and sends to Bob
    3. Bob measures with random bases
    4. Alice and Bob compare bases (classical channel)
    5. They keep bits where bases matched
    6. Error checking and privacy amplification
    """
    
    def __init__(self, num_bits=100):
        self.num_bits = num_bits
        self.alice_bits = []
        self.alice_bases = []
        self.bob_bases = []
        self.bob_results = []
        self.shared_key = []
        
    def alice_prepare_qubits(self):
        """Alice generates random bits and bases, then prepares qubits"""
        self.alice_bits = np.random.randint(0, 2, self.num_bits)
        self.alice_bases = np.random.randint(0, 2, self.num_bits)  # 0: Z-basis, 1: X-basis
        print(f"Alice's bits: {self.alice_bits[:20]}...")
        print(f"Alice's bases: {self.alice_bases[:20]}...")
        
    def bob_choose_bases(self):
        """Bob randomly chooses measurement bases"""
        self.bob_bases = np.random.randint(0, 2, self.num_bits)
        print(f"Bob's bases: {self.bob_bases[:20]}...")
            
    def simulate_quantum_channel(self, error_rate=0.0):
        """Simulate sending qubits from Alice to Bob"""
        simulator = AerSimulator()

        self.bob_results = []

        for i in range(self.num_bits):
            # Create quantum circuit
            qc = QuantumCircuit(1, 1)
            
            # Alice encodes her bits
            if self.alice_bits[i] == 1:
                qc.x(0)  # Flip to |1>
                
            # Alice applies basis rotation
            if self.alice_bases[i] == 1:  # X-basis
                qc.h(0)
                
            # Simulate channel noise
            if np.random.random() < error_rate:
                qc.x(0)  # Bit flip error
                
            # Bob measures in his chosen basis
            if self.bob_bases[i] == 1:  # X-basis
                qc.h(0)
                
            qc.measure(0, 0)
            
            # Execute circuit
            compiled_circuit = transpile(qc, simulator)
            result = simulator.run(compiled_circuit, shots=1).result()
            counts = result.get_counts()
            measured_bit = int(list(counts.keys())[0])
            self.bob_results.append(measured_bit)
            
        print(f"Bob's measurements: {self.bob_results[:20]}...")

    def simulate_caesar_intercept_resend(self, error_rate=0.0, intercept_rate=1.0):
        simulator = AerSimulator()

        self.bob_results = []

        for i in range(self.num_bits):
            # The first circuit, coming directly from Alice.
            direct_circuit = QuantumCircuit(1, 1)

            # Alice does her initial encoding and basis rotation.
            if self.alice_bits[i] == 1:
                direct_circuit.x(0)  # Flip to |1>

            if self.alice_bases[i] == 1: # X-basis
                direct_circuit.h(0)

            # Does Caeser actually interrupt this one?
            if np.random.random() < intercept_rate:
                # Caesar uses his basis to measure
                caesar_base = np.random.randint(0, 2)
                
                if caesar_base == 1:
                    direct_circuit.h(0)
    
                direct_circuit.measure(0,0)
    
                # Execute eavesdropping measurements
                compiled_circuit = transpile(direct_circuit, simulator)
                result = simulator.run(compiled_circuit, shots=1).result()
                counts = result.get_counts()
                measured_bit = int(list(counts.keys())[0])
                caesar_bit = measured_bit
    
                # Caesar sends bits down the line for Bob
                resend_circuit = QuantumCircuit(1,1)
    
                # Prepare the state Caesar though he measured
                if caesar_bit == 1:
                    resend_circuit.x(0) # Flip to |1>
    
                if caesar_base == 1:
                    resend_circuit.h(0)
    
                # Bob measures with his random basis
                if self.bob_bases[i] == 1:
                    resend_circuit.h(0)
    
                resend_circuit.measure(0,0)
    
                # Execute interrupted circuit
                compiled_circuit = transpile(resend_circuit, simulator)
                result = simulator.run(compiled_circuit, shots=1).result()
                counts = result.get_counts()
                measured_bit = int(list(counts.keys())[0])
                self.bob_results.append(measured_bit)

            else:
                # Simulate channel noise
                if np.random.random() < error_rate:
                    direct_circuit.x(0)  # Bit flip error
                    
                # Bob measures in his chosen basis
                if self.bob_bases[i] == 1:  # X-basis
                    direct_circuit.h(0)
                    
                direct_circuit.measure(0, 0)
                
                # Execute circuit
                compiled_circuit = transpile(direct_circuit, simulator)
                result = simulator.run(compiled_circuit, shots=1).result()
                counts = result.get_counts()
                measured_bit = int(list(counts.keys())[0])
                self.bob_results.append(measured_bit)

        print(f"Bob's measurements: {self.bob_results[:20]}...")
        
    def sift_key(self):
        self.shared_key = []
        
        """Alice and Bob compare bases and keep matching ones"""
        for i in range(self.num_bits):
            if self.alice_bases[i] == self.bob_bases[i]:
                self.shared_key.append(self.bob_results[i])
                
        print(f"\nBases matched: {len(self.shared_key)}/{self.num_bits}")
        print(f"Shared key (first 20): {self.shared_key[:20]}...")
        
    def estimate_error_rate(self, sample_size=None):
        """Estimate quantum bit error rate (QBER)"""
        if sample_size is None:
            sample_size = min(len(self.shared_key) // 2, 50)
            
        errors = 0
        alice_sample = []
        
        # Get Alice's bits that correspond to shared key
        key_index = 0
        for i in range(self.num_bits):
            if self.alice_bases[i] == self.bob_bases[i]:
                if key_index < sample_size:
                    alice_sample.append(self.alice_bits[i])
                    if self.alice_bits[i] != self.shared_key[key_index]:
                        errors += 1
                key_index += 1
                
        qber = errors / sample_size if sample_size > 0 else 0
        print(f"\nQBER: {qber:.2%} ({errors}/{sample_size} errors)")
        
        # Remove sampled bits from key
        self.shared_key = self.shared_key[sample_size:]
        return qber
        
    def run_protocol(self, error_rate=0.0, runs=1):
        """Execute the complete BB84 protocol"""
        print("=" * 50)
        print("BB84 Quantum Key Distribution Protocol")
        print("=" * 50)

        print("-" * 25)
        print("First test: Direct Circuit")
        print(f"Passive Error rate: {error_rate}")
        print("-" * 25)

        qber_list = []
        for i in range(runs):
            print(f"[Run {i+1}]")
            self.alice_prepare_qubits()
            self.bob_choose_bases()
    
            self.simulate_quantum_channel(error_rate)
    
            self.sift_key()
            qber = self.estimate_error_rate()
    
            print(f"Final shared key length: {len(self.shared_key)} bits")
    
            print(f"Secure key established: {len(shared_key)} bits\n")

            qber_list.append(qber)

        print(f"Average QBER Error Rate: {(sum(qber_list) / float(len(qber_list))):.2%}\n")

        print("-" * 25)
        print("Second test: Caesar the Interceptor")
        print("-" * 25)

        rates = [0.25, 0.5, 0.75, 1.0]

        for intercept_rate in rates:
            print("<>" * 10)
            print(f"Interception rate: {intercept_rate}")
            print("<>" * 10)
            
            qber_list = []
            for i in range(runs):
                print(f"[Run {i+1}]")
                self.alice_prepare_qubits()
                self.bob_choose_bases()
                
                self.simulate_caesar_intercept_resend(error_rate,intercept_rate)
                
                self.sift_key()
                qber = self.estimate_error_rate()
        
                print(f"Final shared key length: {len(self.shared_key)} bits")
        
                print(f"Secure key established: {len(shared_key)} bits\n")
    
                qber_list.append(qber)
    
            print(f"Average QBER Error Rate: {(sum(qber_list) / float(len(qber_list))):.2%}\n")


# Example usage
if __name__ == "__main__":
    # Run BB84 protocol
    bb84 = BB84Protocol(num_bits=128)
    bb84.run_protocol(error_rate=0.0,runs=5)

BB84 Quantum Key Distribution Protocol
-------------------------
First test: Direct Circuit
Passive Error rate: 0.0
-------------------------
[Run 1]
Alice's bits: [0 1 0 1 0 1 1 1 0 1 0 0 0 0 0 1 1 1 1 1]...
Alice's bases: [0 0 1 1 0 1 0 0 0 1 0 0 1 0 1 0 0 0 0 1]...
Bob's bases: [0 1 1 0 1 1 1 1 0 0 1 1 1 0 1 0 0 1 1 1]...
Bob's measurements: [0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1]...

Bases matched: 57/128
Shared key (first 20): [0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1]...

QBER: 0.00% (0/28 errors)
Final shared key length: 29 bits
Secure key established: 116 bits

[Run 2]
Alice's bits: [1 0 1 1 1 0 1 1 1 0 0 0 1 0 1 1 0 1 0 1]...
Alice's bases: [0 1 0 0 1 1 1 0 1 1 0 1 0 0 1 1 0 0 0 1]...
Bob's bases: [1 1 0 1 0 1 1 0 0 0 0 0 0 1 1 1 1 1 1 1]...
Bob's measurements: [0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1]...

Bases matched: 68/128
Shared key (first 20): [0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0]...

QBER: